In [7]:
from sklearn.metrics import accuracy_score, roc_curve, confusion_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from tabpfn import TabPFNClassifier
from sklearn.preprocessing import StandardScaler
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
from natsort import natsorted
from tqdm import tqdm
import gc
gc.collect()
torch.cuda.empty_cache()
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"

In [8]:
def metrics(tn, fp, fn, tp):
    """
    Calculate confusion matrix and various performance metrics.
    Args:
        tn (int): True Negatives
        fp (int): False Positives
        fn (int): False Negatives
        tp (int): True Positives
    Returns:
        cm_df (pd.DataFrame): Confusion matrix as a DataFrame.
        metric_df (pd.DataFrame): Performance metrics as a DataFrame.
    """
    tn = tn.astype(np.float32)
    fp = fp.astype(np.float32)
    fn = fn.astype(np.float32)
    tp = tp.astype(np.float32)

    cm_dict = {'predict\\actual':['Positive', 'Negative']
               ,'Positive':[tp, fn]
               ,'Negative':[fp, tn]}

    cm_df = pd.DataFrame(cm_dict)
    
    accuracy = round((tp + tn) / (tp + tn + fp + fn), 4) if (tp + tn + fp + fn) > 0 else 0  
    sensitivity = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0
    specificity = round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0
    ppv = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0
    npv = round(tn / (tn + fn), 4) if (tn + fn) > 0 else 0
    mcc = round((tp * tn - fp * fn) / np.sqrt(float(tp + fp) * float(tp + fn) * float(tn + fp) * float(tn + fn)), 4) if (tp + fp) > 0 and (tp + fn) > 0 and (tn + fp) > 0 and (tn + fn) > 0 else 0

    metric_dict = {'metric':['Accuruacy', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'MCC'],
                   'Value':[accuracy, sensitivity, specificity, ppv, npv, mcc]}

    metric_df = pd.DataFrame(metric_dict)

    return cm_df, metric_df

In [9]:
dataset_dir = r'V:\dunwei\MACE\dataset\TabPFN_result\sr10k_500_1k_iSKNA_5_10min_1kpts_60win1stride_unuse_trH80P126win_teH126P126win_tsfresh_4PCA_5static_split\dataset/'
save_data_dir = r"V:\dunwei\MACE\dataset\TabPFN_result\sr10k_500_1k_iSKNA_5_10min_1kpts_60win1stride_unuse_trH80P126win_teH126P126win_tsfresh_4PCA_5static_split/"
save_combine_data_dir = save_data_dir + "combine/"

In [10]:
for save_dir in [save_data_dir, save_combine_data_dir]:
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f'create directory: {save_dir}')
    else:
        print(f'directory already exists: {save_dir}')

directory already exists: V:\dunwei\MACE\dataset\TabPFN_result\sr10k_500_1k_iSKNA_5_10min_1kpts_60win1stride_unuse_trH80P126win_teH126P126win_tsfresh_4PCA_5static_split/
create directory: V:\dunwei\MACE\dataset\TabPFN_result\sr10k_500_1k_iSKNA_5_10min_1kpts_60win1stride_unuse_trH80P126win_teH126P126win_tsfresh_4PCA_5static_split/combine/


In [11]:
all_data_dict = np.load(os.path.join(dataset_dir, 'all_data_dict.npz'), allow_pickle=True)
non_mace_per_windows_df = pd.read_csv(os.path.join(dataset_dir, 'non_mace_per_windows.csv'))
mace_per_windows_df = pd.read_csv(os.path.join(dataset_dir, 'mace_per_windows.csv'))
print(non_mace_per_windows_df.head())
print(mace_per_windows_df.head())
print(all_data_dict['MACE001'].shape)
print(all_data_dict['MACE021'].shape)

  research_id  window_per_person
0     MACE001                241
1     MACE002                241
2     MACE003                241
3     MACE004                241
4     MACE005                241
  research_id  window_per_person
0     MACE021                241
1     MACE028                241
2     MACE033                241
3     MACE044                241
4     MACE046                241
(241, 10)
(241, 10)


In [ ]:
gc.collect()
torch.cuda.empty_cache()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
GROUP_IDS = np.array(list(all_data_dict.files))
GROUP_LABELS = np.concatenate([np.zeros(len(non_mace_per_windows_df['research_id'].to_numpy())), np.ones(len(mace_per_windows_df['research_id'].to_numpy()))])

total_train_tp = total_train_fn = total_train_fp = total_train_tn = 0
total_test_tp = total_test_fn = total_test_fp = total_test_tn = 0
total_valid_tp = total_valid_fn = total_valid_fp = total_valid_tn = 0

for fold, (train_idx, test_idx) in enumerate(skf.split(GROUP_IDS, GROUP_LABELS)):

    train_list = []
    test_list = []
    train_ids = []
    test_ids = []

    train_group_ids = GROUP_IDS[train_idx]
    test_group_ids = GROUP_IDS[test_idx]

    for train_id in train_group_ids:
        if train_id in non_mace_per_windows_df['research_id'].to_numpy():
            data = all_data_dict[train_id][:10, :].copy()
        elif train_id in mace_per_windows_df['research_id'].to_numpy():
            data = all_data_dict[train_id][:40, :].copy()
        train_list.append(data)
        train_ids.extend([str(train_id)] * data.shape[0])

    for test_id in test_group_ids:
        if test_id in non_mace_per_windows_df['research_id'].to_numpy():
            data = all_data_dict[test_id][:, :].copy()
        elif test_id in mace_per_windows_df['research_id'].to_numpy():
            data = all_data_dict[test_id][:, :].copy()
        test_list.append(data)
        test_ids.extend([str(test_id)] * data.shape[0])
    
            
    train = np.vstack(train_list)
    train_ids_df = pd.DataFrame({'research_id': train_ids})
    test = np.vstack(test_list)
    test_ids_df = pd.DataFrame({'research_id': test_ids})
    print(f'Train shape : {train.shape}, Test shape : {test.shape}')

    X_train, y_train = train[:, 1:], train[:, 0]
    X_test, y_test = test[:, 1:], test[:, 0]

    clf = TabPFNClassifier(device='auto', random_state=42)
    clf.fit(X_train, y_train)

    gc.collect()
    torch.cuda.empty_cache()

    yhat_train = clf.predict(X_train)
    yaht_train_proba = clf.predict_proba(X_train)[:, 1]

    gc.collect()
    torch.cuda.empty_cache()

    yhat_test = clf.predict(X_test)
    yhat_test_proba = clf.predict_proba(X_test)[:, 1]

    gc.collect()
    torch.cuda.empty_cache()

    y_train_df = pd.DataFrame(y_train, columns=['y_train'])
    yhat_train_df = pd.DataFrame(yhat_train, columns=['yhat_train'])
    yhat_test_df = pd.DataFrame(yhat_test, columns=['yhat_test'])
    y_test_df = pd.DataFrame(y_test, columns=['y_test'])
    yhat_train_proba_df = pd.DataFrame(yaht_train_proba, columns=['yhat_train_proba'])
    yhat_test_proba_df = pd.DataFrame(yhat_test_proba, columns=['yhat_test_proba'])
    y_train_label = pd.concat([train_ids_df.reset_index(drop=True), y_train_df.reset_index(drop=True), yhat_train_df], axis=1)
    y_test_label = pd.concat([test_ids_df.reset_index(drop=True), y_test_df.reset_index(drop=True), yhat_test_df], axis=1)
    yhat_train_probability = pd.concat([train_ids_df.reset_index(drop=True), yhat_train_proba_df], axis=1)
    yhat_test_probability = pd.concat([test_ids_df.reset_index(drop=True), yhat_test_proba_df], axis=1)

    y_train_label.to_csv(os.path.join(save_data_dir, f'y_train_label_{fold + 1}.csv'), index=False)
    y_test_label.to_csv(os.path.join(save_data_dir, f'y_test_label_{fold + 1}.csv'), index=False)
    yhat_train_probability.to_csv(os.path.join(save_data_dir, f'yhat_train_probability_{fold + 1}.csv'), index=False)
    yhat_test_probability.to_csv(os.path.join(save_data_dir, f'yhat_test_probability_{fold + 1}.csv'), index=False)

    train_tn, train_fp, train_fn, train_tp = confusion_matrix(y_train, yhat_train).ravel()
    total_train_tp += train_tp
    total_train_fn += train_fn
    total_train_fp += train_fp
    total_train_tn += train_tn

    test_tn, test_fp, test_fn, test_tp = confusion_matrix(y_test, yhat_test).ravel()
    total_test_tp += test_tp
    total_test_fn += test_fn
    total_test_fp += test_fp
    total_test_tn += test_tn

train_cm_df, train_metric_df = metrics(total_train_tn, total_train_fp, total_train_fn, total_train_tp)
test_cm_df, test_metric_df = metrics(total_test_tn, total_test_fp, total_test_fn, total_test_tp)
train_cm_df.to_csv(save_combine_data_dir + f"train_cm.csv", index=False)
train_metric_df.to_csv(save_combine_data_dir + f"train_metric.csv", index=False)
test_cm_df.to_csv(save_combine_data_dir + f"test_cm.csv", index=False)
test_metric_df.to_csv(save_combine_data_dir + f"test_metric.csv", index=False)

print('===========   TRAIN RESULTS   ===========')
print(f"confusion matrix of train :\n{train_cm_df.to_string(index=False)}\n")
print(f"metric of train :\n{train_metric_df.to_string(index=False)}\n")


print('===========   TEST RESULTS   ===========')
print(f"confusion matrix of test :\n{test_cm_df.to_string(index=False)}\n")
print(f"metric of test :\n{test_metric_df.to_string(index=False)}\n")


Train shape : (36672, 10), Test shape : (13230, 10)
Train shape : (36672, 10), Test shape : (13230, 10)
Train shape : (36752, 10), Test shape : (13104, 10)
Train shape : (36752, 10), Test shape : (13104, 10)
Train shape : (36752, 10), Test shape : (13104, 10)
===========   TRAIN RESULTS   ===========
confusion matrix of train :
predict\actual  Positive  Negative
      Positive   45179.0     170.0
      Negative     181.0  138070.0

metric of train :
     metric  Value
  Accuruacy 0.9981
Sensitivity 0.9960
Specificity 0.9988
        PPV 0.9963
        NPV 0.9987
        MCC 0.9949

===========   TEST RESULTS   ===========
confusion matrix of test :
predict\actual  Positive  Negative
      Positive    2478.0    6649.0
      Negative    8862.0   47783.0

metric of test :
     metric  Value
  Accuruacy 0.7642
Sensitivity 0.2185
Specificity 0.8778
        PPV 0.2715
        NPV 0.8436
        MCC 0.1053

